In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc hivqe-qiskit-function-utils qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

*اطّلع على [مرجع API](https://docs.quantum.ibm.com/api/functions/qunova-chemistry)*

> **Note:** دوال Qiskit ميزة تجريبية متاحة فقط لمستخدمي خطط IBM Quantum&reg; Premium وFlex وOn-Prem (عبر IBM Quantum Platform API). هي في مرحلة إصدار معاينة وقابلة للتغيير.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>

## نظرة عامة

في الكيمياء الكمومية، تتمحور مسألة البنية الإلكترونية حول إيجاد حلول معادلة شرودنغر الإلكترونية — دوال الموجة الكمومية التي تصف سلوك إلكترونات النظام. هذه الدوال عبارة عن متجهات من السعات المركبة، حيث تقابل كل سعة مساهمة إحدى التهيئات الإلكترونية الممكنة.

الحالة الأرضية هي دالة الموجة ذات أدنى طاقة في النظام، وتحتل مكانة خاصة في دراسة الأنظمة الجزيئية. يأخذ النهج الأدق لحساب الحالة الأرضية بعين الاعتبار جميع التهيئات الإلكترونية الممكنة، إلا أن ذلك يصبح غير قابل للتطبيق على الأنظمة الأكبر إذ يتزايد عدد التهيئات بشكل أسي مع حجم النظام.

يُعدّ الـ HI-VQE (Handover Iterative Variational Quantum Eigensolver) منهجًا هجينًا مبتكرًا بين الحوسبة الكمومية والكلاسيكية، يهدف إلى تقدير دقيق للحالة الأرضية للأنظمة الجزيئية. يدمج المعالجات الكمومية مع الحوسبة الكلاسيكية، مستعينًا بالمعالجات الكمومية لاستكشاف التهيئات الإلكترونية المرشحة بكفاءة، ثم يحسب دالة الموجة الناتجة على الحواسيب الكلاسيكية. من خلال توليد دوال موجة مضغوطة وذات دقة كيميائية عالية، يُعزّز HI-VQE البحث والاكتشاف في كيمياء الكم وعلم المواد.

![صورة توضح نظرة عامة على خوارزمية HI-VQE من Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

يُقلّل HI-VQE من التعقيد الحسابي لمسألة البنية الإلكترونية من خلال تقدير الحالة الأرضية بدقة عالية وكفاءة. يركّز على مجموعة فرعية مختارة بعناية من أبرز التهيئات الإلكترونية، محقّقًا توازنًا مثاليًا بين الدقة والكفاءة.

يجمع HI-VQE بين إمكانيات الحواسيب الكلاسيكية والكمومية، ويُحسّن تقدير دالة الموجة الحالية تدريجيًا وبصورة متكررة. تُسهم تقنياته الفريدة لبناء الفضاء الجزئي في رفع كفاءة اختيار التهيئات، مما يمنح المستخدمين تحكّمًا حسابيًا أكبر ودقة محسّنة في محاكاة كيمياء الكم.

إن كنت تودّ التعمق في فهم الخوارزمية، يمكنك [قراءة الورقة البحثية المرتبطة](https://arxiv.org/abs/2503.06292).

## الوصف

يتزايد عدد التهيئات الإلكترونية لنظام جزيئي بشكل أسي مع حجمه. غير أنه في حالات إلكترونية معينة، كالحالة الأرضية، يشيع أن نسبة صغيرة فقط من التهيئات تُسهم بشكل ملحوظ في طاقة الحالة. تستغل أساليب التفاعل التشكيلي المنتقى (SCI) هذه الندرة لتقليل التكاليف الحسابية عبر تحديد التهيئات الأكثر صلة والتركيز عليها. تُسمّى هذه المجموعة الفرعية من التهيئات بالفضاء الجزئي.

يستثمر HI-VQE الكفاءة الجوهرية للحواسيب الكمومية في تمثيل الأنظمة الجزيئية للمساعدة في البحث عن الفضاء الجزئي. يدمج الروتينات الكلاسيكية والكمومية لحل مسألة البنية الإلكترونية بدقة عالية. على خلاف أساليب SCI الكمومية الموجودة، يجمع HI-VQE بين التدريب التغايري والبناء التكراري للفضاء الجزئي وفرز التهيئات قبل القطرنة، لتعزيز الكفاءة عبر تقليل القياسات الكمومية والتكرارات وتكاليف القطرنة الكلاسيكية. وبالتالي يمكن تطبيق HI-VQE على أنظمة جزيئية أكبر تحتاج إلى مزيد من qubits، مع تقليل تكلفة حل مسألة بحجم معين بالدقة ذاتها.

![صورة توضح وصفًا تفصيليًا لكل خطوة في خوارزمية HI-VQE من Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

لحساب الحالة الأرضية لنظام ما، يبدأ HI-VQE بالاستعانة بحزمة الكيمياء الكلاسيكية PySCF لتوليد تمثيل جزيئي من المدخلات التي يوفّرها المستخدم، كالهندسة الجزيئية وبيانات أخرى عن الجزيء. ثم يدخل في حلقة تحسين هجينة كمومية-كلاسيكية، يُحسّن فيها تدريجيًا فضاءً جزئيًا لتمثيل الحالة الأرضية بشكل أمثل مع تقليل عدد التهيئات المضمّنة. تستمر الحلقة حتى تتحقق معايير التقارب كحجم الفضاء الجزئي أو استقرار الطاقة، ثم يُخرج الخوارزم دالة موجة الحالة الأرضية المحسوبة وطاقتها. يمكن استخدام هذه النتائج لبناء أسطح طاقة الوضع الدقيقة وإجراء مزيد من التحليل للنظام.

تتمحور حلقة التحسين حول ضبط معاملات دائرة كمومية لتوليد فضاء جزئي عالي الجودة. يوفّر HI-VQE ثلاثة خيارات للدائرة الكمومية: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving) و[efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) و[LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). يُهيّأ التحسين قريبًا من حالة مرجعية Hartree-Fock لملاءمتها العامة. تُنفَّذ الدائرة بعد ذلك على جهاز كمومي وتُؤخذ عيّنات من التهيئات من الحالة الكمومية الناتجة وتُعاد كسلاسل ثنائية. نتيجةً لضوضاء الجهاز الكمومي، قد تكون بعض التهيئات المُعيَّنة غير صالحة فيزيائيًا، إذ لا تحفظ عدد الإلكترونات أو الدوران. يتعامل HI-VQE مع هذا باستخدام عملية استرداد التهيئة من حزمة [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview)، ليتمكن المستخدمون إما من تصحيح التهيئات غير الصالحة أو تجاهلها.

تخضع التهيئات الصالحة بعد ذلك لخطوة فرز اختيارية لإزالة تلك المتوقع أن تُسهم بالحد الأدنى. يُقلّل ذلك من بُعد الفضاء الجزئي وبالتالي يخفض تكلفة خطوة القطرنة. في حال تفعيل الفرز، يُبنى Hamiltonian فضاء جزئي أوّلي من التهيئات الصالحة ويُجرى قطرنة بمعايير إنهاء فضفاضة جدًا. رغم أن دقة السعات الناتجة لكل تهيئة منخفضة، إلا أنها فعّالة في التنبؤ بالتهيئات التي ينبغي استبعادها من الفضاء الجزئي في هذه الجولة، وتُحسب بسرعة.

تُضاف التهيئات المختارة إلى الفضاء الجزئي ويُسقَط Hamiltonian النظام في هذا الفضاء. يتحدث الفضاء الجزئي تكراريًا محافظًا على أبرز التهيئات عبر الجولات. يتميز هذا النهج عن البدائل بأن الدائرة الكمومية لا تحتاج إلى تقريب الحالة الأرضية الكاملة في كل خطوة.

بعد ذلك، يُقطَّر Hamiltonian الفضاء الجزئي كلاسيكيًا للحصول على أدنى قيمة ذاتية ومتجهها الذاتي المقابل، وهما تقريب للحالة الأرضية وطاقتها. مع تحسّن جودة الفضاء الجزئي عبر الجولات، يقترب حساب الحالة الأرضية من الحالة الحقيقية. يمكن إجراء خطوة فرز إضافية في هذه المرحلة لإزالة التهيئات ذات المساهمة غير الجوهرية في الحالة الأرضية المحسوبة. تضمن هذه الخطوة أن الفضاء الجزئي الذي يُنقل للجولة التالية يكون بالحجم الأمثل. ويُقيَّم ذلك بناءً على السعات التي تُعيدها عملية القطرنة، إذ تمثّل أهمية مساهمة كل تهيئة في الحالة الأرضية المحسوبة.

ثم يُجرى فحص تقارب لتحديد ما إذا كانت التكرارات الإضافية ستحسّن النتائج. إن كان الأمر كذلك، تُجرى خطوة توسيع كلاسيكية اختيارية، تُحدَّث معاملات الدائرة الكمومية لمزيد من تقليل الطاقة المحسوبة، وتتكرر العملية. تُولّد خطوة التوسيع الكلاسيكية تهيئات إضافية للفضاء الجزئي تكمّل ما أُخذ عيّنات منه من الجهاز الكمومي. تُحدّد أولًا التهيئة ذات أكبر سعة في نتائج القطرنة، ثم تُولّد تهيئات جديدة بإثارات أحادية وثنائية من التهيئة المحدّدة، وتُضاف العدد المطلوب من هذه التهيئات إلى الفضاء الجزئي.

بمجرد التحقق من تقارب الجولات، يُعيد HI-VQE الحالة الأرضية المحسوبة (على شكل الحالات في الفضاء الجزئي وسعاتها في دالة الموجة)، وطاقتها، ومقياس تباين الطاقة الذي يشير إلى ما إذا كانت الحالة المحسوبة تُشكّل حالة ذاتية لـ Hamiltonian النظام.

يستطيع المستخدمون تحديد الدائرة الكمومية المستخدمة وعدد اللقطات لكل دائرة، إضافةً إلى التحكم في حجم الفضاء الجزئي أو تفعيل التوليد الكلاسيكي لتهيئات إضافية لدعم تلك المولّدة كميًا. بهذه الطريقة، يمكن للمستخدمين تكييف سلوك HI-VQE لتناسب تطبيقاتهم المطلوبة.

## الترخيص

يُرجى ملاحظة أن استخدام دالة Qiskit هذه مقيّد بالمسائل التي تتطلب 20 qubit كحد أقصى، ما لم يُحصل على ترخيص يمنح حدًا أعلى.

يُرجى مراسلة [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) إن كنت ترغب في الاستفسار عن الحصول على ترخيص.

## البدء

أولًا، [اطلب الوصول إلى الدالة](https://forms.office.com/r/zN3hvMTqJ1).
ثم قم بالمصادقة باستخدام [مفتاح IBM Quantum&reg; API](http://quantum.cloud.ibm.com/) الخاص بك، وبافتراض أنك [حفظت حسابك](/guides/functions#install-qiskit-functions-catalog-client) في بيئتك المحلية، اختر دالة Qiskit كما يلي:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

يمكن تعريف خيارات إضافية وتوفيرها للنظام الجزيئي بصيغة القاموس التالية.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

نفّذ الدالة مع مدخلات الهندسة والخيارات.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

من المفيد طباعة معرّف مهمة الدالة حتى يمكن تقديمه في طلبات الدعم إذا حدث خطأ ما.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


يستخدم هذا المثال 16 qubit مع 8 مدارات من أساس sto3g لجزيء NH3.
تحقق من [الحالة](/guides/functions#check-job-status) أو استرجع [النتائج](/guides/functions#retrieve-results) لعمل Qiskit Function الخاص بك على النحو التالي:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824448589364075, 0.009527106392132133, 6.854074372058527e-08, 3.591500190038039e-07, 0.0012975231577544268, 2.310159709002111e-05, ...], 'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, ...}


بعد اكتمال المهمة، يمكن الحصول على النتائج باستخدام نسخة `result()`.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa
Sampled Number of States: 1936


## Licensing

Note that use of this Qiskit Function is limited to problems requiring at most 20 qubits, unless a license is obtained granting a higher limit.

Email [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) with license inquiries.

### Example of licensed function use

Licensed users are issued a license token, and then must use a wrapper library to submit their license token to the function.
The wrapper library can be installed [from PyPI](https://pypi.org/project/hivqe-qiskit-function-utils/) with `pip install hivqe-qiskit-function-utils`.
The below example shows how this library should be used to submit your license token when calling the function.

In [ ]:
import math
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from hivqe_qiskit_function_utils import FunctionWrapper

catalog = QiskitFunctionsCatalog(
    token="your_qiskit_functions_catalog_token",
    channel="ibm_quantum_platform",
    instance="your_ibm_instance",
)

molecule_geometry = f"""
O 0 0 0;
H {-0.957*math.sin(math.radians(104.5)/2.0)} {0.957*math.cos(math.radians(104.5)/2.0)} 0;
H {0.957*math.sin(math.radians(104.5)/2.0)} {0.957*math.cos(math.radians(104.5)/2.0)} 0
"""

hivqe = FunctionWrapper(
    token="your_hivqe_license_token",
    function=catalog.load("qunova/hivqe-chemistry"),
)
job = hivqe.run(
    geometry=molecule_geometry,
    backend_name="ibm_torino",
    max_states=10000,
    max_expansion_states=1000,
    hivqe_options={"ansatz": "epa", "max_iter": 10},
)
result = job.result()

للوصول إلى طاقة الحالة الأرضية، استخدم مفتاح "energy". يوفّر مفتاح "eigenvector" معاملات CI مع تدوين السلسلة الثنائية المقابلة للتهيئة الإلكترونية المخزّنة في "states" من النتائج.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

## الأداء
يعرض هذا القسم حسابات المعايير التوضيحية لـ HI-VQE مع حالة بـ 24 qubit لجزيء Li2S، وحالة بـ 40 qubit لجزيء N2، وحالة بـ 44 qubit لنظام FeP-NO.
#### منحنى سطح طاقة الوضع للتفكك لجزيء Li2S بـ 24 qubit
يُعرض منحنى PES مع مرجع FCI والتخمين الأوّلي من RHF، إضافةً إلى خطأ الطاقة من مرجع FCI.

![صورة توضح أن HI-VQE ينتج حلولًا ضمن الدقة الكيميائية لمنحنى PES المرجعي الكلاسيكي لنظام Li2S.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

أُجريت الحسابات بالهندسات والخيارات التالية.

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

تمثّل النقاط الحمراء نتائج حسابات HI-VQE لستة هندسات مختلفة، وثلاث هندسات تتوافق مع 1.51 و2.40 و3.80 أنغستروم مُقدَّمة كمدخلات في الخلية أعلاه.
#### منحنى PES للتفكك لجزيء N2 بـ 40 qubit
تم التعرف على جزيء النيتروجين كنظام متعدد المراجع مع مساهمات طاقة ارتباط كبيرة تتجاوز حالة Hartree-Fock. أجرينا حسابًا معياريًا لجزيء N2 مع أساس cc-pvdz، (20o,14e) باستخدام اختيار المدار النشط homo-lumo. رقم الفضاء النشط الكامل (CAS) لتمثيل هذه المسألة هو 6,009,350,400. من غير الممكن الحصول على حل مسألة القيم الذاتية (للطاقة والبنية الإلكترونية) بهذا العدد من الحالات باستخدام حاسوب مكتبي قوي (16cpu/64GB). مع HI-VQE، يستطيع المستخدمون البحث بكفاءة في الفضاء الجزئي لحالات CAS للحصول على نتائج دقيقة كيميائيًا مع توفير موارد الحوسبة بشكل ملحوظ. تُظهر الرسوم البيانية التالية منحنى PES لحساب HI-VQE بـ 40 qubit لتفكك جزيء N2.

![صورة توضح أن HI-VQE ينتج حلولًا ضمن الدقة الكيميائية لمنحنى PES المرجعي الكلاسيكي لنظام N2.](../docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)
#### منحنى PES للتفكك لـ iron(II)-porphyrin خماسي التنسيق مع نظام NO بـ 44 qubit
نظام كيميائي مثير للاهتمام آخر هو مركّب iron(II)-porphyrin (FeP) مع ليغاند nitric oxide (NO) منسّق، وهو نظام metalloporphyrin ذو صلة بيولوجية يؤدي أدوارًا محورية في عمليات فسيولوجية متعددة. في هذا المثال، استُخدم HI-VQE لتقدير منحنى سطح طاقة الوضع الدقيق للتفاعل بين جزيئيّ FeP وNO (طاقة الحالة الأرضية لهندسات بمسافات مختلفة). يحتوي النظام المدمج على 450 مدارًا و202 إلكترونًا (450o,202e) مع أساس 6-31g(d) إجمالًا. اُستخدم اختيار المدار النشط homo-lumo لحساب الحالة الأصغر من الحالة الحقيقية بـ (22o,22e). من نتائج المعايير التالية، تمكّنا من تحقيق الدقة الكيميائية (>&nbsp;1.6 mHa) مع حساب كيمياء الحاسوب الكلاسيكي المتقدم CASCI(DMRG) (22o,22e) كمرجع.

![صورة توضح أن HI-VQE ينتج حلولًا ضمن الدقة الكيميائية لمنحنى PES المرجعي الكلاسيكي لنظام FeP-NO.](../docs/images/guides/qunova-chemistry/fepno_44qubits.avif)
## المعايير
- حجم المصفوفة الدقيق هو عدد المحدِّدات للحل الدقيق، مثل FCI وCASCI.
- يأخذ حساب HI-VQE عيّنات من الفضاء الجزئي ويحسبه (أي حجم مصفوفة HI-VQE).
- الوقت الإجمالي يشمل وقت تشغيل QPU وتشغيلات Qiskit Function مع CPU.
- تُقدَّر الدقة من فارق الطاقة عن الحل الدقيق.

|   النظام الكيميائي  | عدد الـ qubits | حجم المصفوفة الدقيق |  حجم مصفوفة HI-VQE  | E(diff) من الدقيق (mHa) | عدد الجولات | الوقت الإجمالي    | استخدام وقت تشغيل QPU |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37 s          | 34 s              |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250 s         | 50 s              |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354 s         | 54 s              |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531 s        | 121 s             |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174 s        | 130 s             |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547 s       | 258 s             |
## استرجاع رسائل الخطأ
إذا فشل عملك، ستكون الحالة `ERROR` واستدعاء `job.result()` سيُطلق استثناء: